# Honest Full-Precision Eval — Path B

**Цель**: устранить inference-artifact в сравнении base vs GSPO vs KTO. Все три модели прогоняются через Hugging Face transformers в bf16 с **identical decoding protocol** + **identical Combined Judge** (Cerebras).

**Compute**: RTX 6000 Ada (48GB), ~8.71 units/hour. Estimated: 3 models × 143 calc problems × ~30s = ~3.6h = ~31 units (≪ 600).

**Output**: `evaluation/reports/honest_full_precision_2026-04-30.json` со строгим apple-to-apple сравнением.

**Структура**:
1. Setup (install training stack → **RESTART RUNTIME** → imports/.env)
2. **DECODING_CONFIG** — TODO(human): главное методологическое решение
3. Load eval dataset (143 calc problems)
4. Per-model inference function (base / +GSPO adapter / +KTO adapter)
5. Combined Judge (Cerebras, единый для всех трёх)
6. Run + save

In [1]:
# Cell 1: Setup (Path A — replicate training stack exactly)
# First run: install training-matched stack → RESTART RUNTIME → re-run this cell.
# Sentinel /content/.install_done_v3 gates install; after restart imports load.
import subprocess, os

# ─── GPU check (need ≥24GB for 9B bf16) ────────────────────
gpu_info = subprocess.check_output(
    ['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader']
).decode().strip()
print(f'GPU: {gpu_info}')
gpu_memory_mib = int(gpu_info.split(',')[1].strip().split()[0])
assert gpu_memory_mib >= 24_000, (
    f'Insufficient VRAM for 9B bf16: {gpu_memory_mib} MiB. Need >=24GB.'
)
print(f'VRAM check passed: {gpu_memory_mib} MiB ({gpu_memory_mib/1024:.1f} GiB)')

# ─── Drive mount + repo clone (idempotent) ────────────────
from google.colab import drive
drive.mount('/content/drive')
if not os.path.exists('/content/MITS'):
    !git clone https://github.com/Siesher/MIST.git /content/MITS
%cd /content/MITS
!git checkout 019-ns-vstar-dpo && git pull origin 019-ns-vstar-dpo

# ─── Install (FIRST RUN ONLY — gated by sentinel) ─────────
# Stack matches grpo_qwen3.5_9b.ipynb cell 1 (proven working on Colab).
SENTINEL = '/content/.install_done_v3'
if not os.path.exists(SENTINEL):
    print('=== Installing training-matched stack (torch 2.10 locked) ===')
    print('  Unsloth 2026.5.1 requires torch<2.11; Colab default 2.10 уже OK.')

    # Step 0: Lock pytorch ecosystem to torch 2.10 — Colab default. Это
    # предотвращает upgrade-to-2.11 от transitive deps трансформеров.
    !pip install -q --upgrade "torch>=2.10,<2.11" "torchvision" "torchaudio>=2.10,<2.11"

    # Step 1: Unsloth core (без force-reinstall — fresh runtime, deps clean)
    !pip install -q --upgrade unsloth unsloth_zoo "torch<2.11"

    # Step 2: transformers v5 — torch upper bound prevents re-upgrade
    !pip install -q --upgrade "transformers>=5.0.0,<6.0" "torch<2.11" trl peft datasets

    # Step 3: scipy + torchvision compat already locked в Step 0
    !pip install -q --upgrade scipy

    # Step 4: Other deps (Cerebras client requires python-dotenv + openai)
    !pip install -q accelerate bitsandbytes sentencepiece protobuf loguru python-dotenv openai
    !pip install -q sympy chempy

    # Step 5: FLA — Triton-based DeltaNet kernels (24/32 layers in Qwen3.5-9B)
    !pip install -q flash-linear-attention 2>&1 | tail -3

    # Step 6: KILL torchcodec. Если torch ends up at 2.11+cu130 (Colab default
    # пытается upgrade), torchcodec wheel ABI mismatch → RuntimeError при dlopen
    # libtorchcodec_core{4..8}.so. sentence_transformers/base/modality_types.py
    # ловит только (ImportError, OSError), не RuntimeError → cascade ломает
    # unsloth import. Uninstall переводит fail в ImportError (catchable).
    print('Removing torchcodec (sentence_transformers fallback handles ImportError gracefully)...')
    !pip uninstall -y torchcodec 2>&1 | tail -1

    # Sentinel: использует stdlib open() — НЕ требует Path import (он только
    # в post-restart section ниже).
    open(SENTINEL, 'w').close()
    print('=' * 60)
    print('  Install complete. RESTART RUNTIME NOW:')
    print('    Runtime → Restart session')
    print('  After restart: re-run THIS cell — install will skip.')
    print('=' * 60)
    raise SystemExit('Restart required — re-run this cell after Runtime → Restart session.')

# ─── Post-restart imports ─────────────────────────────────
# HOTFIX for PIL._typing._Ink (Pillow 12 + torchvision pre-0.23 incompatibility)
try:
    from PIL import _typing
    if not hasattr(_typing, '_Ink'):
        import sys
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade',
                              'torchvision>=0.23.0', 'pillow<12.0.0'])
except ImportError:
    pass

import sys, json, time, logging
from pathlib import Path
from datetime import datetime
from typing import Any, Dict, List

import torch
import transformers
import huggingface_hub
from unsloth import FastLanguageModel
from peft import PeftModel
from dotenv import load_dotenv

# Sanity: transformers 5.x required for qwen3_5
print(f'transformers: {transformers.__version__} | huggingface_hub: {huggingface_hub.__version__} | torch: {torch.__version__}')
assert transformers.__version__.startswith('5.'), (
    f'transformers must be 5.x for Qwen3.5 (model_type=qwen3_5). '
    f'Current: {transformers.__version__}. Restart runtime if just installed.'
)

PROJECT_ROOT = Path('/content/MITS')
sys.path.insert(0, str(PROJECT_ROOT))

# Load .env from Drive (Cerebras keys + optional HF_TOKEN)
ENV_PATH = Path('/content/drive/MyDrive/MITS_secrets/.env')
if ENV_PATH.exists():
    load_dotenv(ENV_PATH)
    print(f'Loaded env from {ENV_PATH}')
    if os.environ.get('HF_TOKEN'):
        from huggingface_hub import login as hf_login
        hf_login(token=os.environ['HF_TOKEN'])
        print('HF authenticated via .env')
    else:
        print('Warning: HF_TOKEN missing in .env — adapter download may 401 if private.')
else:
    print(f'No .env at {ENV_PATH}. Cerebras judge will fail in Phase 0b.')

from training.scripts.evaluate_stage import (
    SYSTEM_PROMPT_CALC,
    extract_answer,
    check_format_compliance,
    evaluate_combined_quality,
    load_eval_dataset,
)
from training.cerebras_client import CerebrasClient

logging.basicConfig(level=logging.INFO, format='%(asctime)s | %(levelname)s | %(message)s')
logger = logging.getLogger('honest_eval')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
logger.info(f'Device: {DEVICE} | bf16: {torch.cuda.is_bf16_supported()}')


GPU: NVIDIA A100-SXM4-40GB, 40960 MiB
VRAM check passed: 40960 MiB (40.0 GiB)
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/MITS
Already on '019-ns-vstar-dpo'
Your branch is up to date with 'origin/019-ns-vstar-dpo'.
From https://github.com/Siesher/MIST
 * branch            019-ns-vstar-dpo -> FETCH_HEAD
Already up to date.


/tmp/ipykernel_1698/986786934.py:89: UserWarning: WARNING: Unsloth should be imported before [transformers] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from unsloth import FastLanguageModel


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


ERROR:bitsandbytes.cextension:bitsandbytes library load error: libnvJitLink.so.13: cannot open shared object file: No such file or directory
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/bitsandbytes/cextension.py", line 320, in <module>
    lib = get_native_library()
          ^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/bitsandbytes/cextension.py", line 298, in get_native_library
    dll = ct.cdll.LoadLibrary(str(binary_path))
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/ctypes/__init__.py", line 460, in LoadLibrary
    return self._dlltype(name)
           ^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/ctypes/__init__.py", line 379, in __init__
    self._handle = _dlopen(self._name, mode)
                   ^^^^^^^^^^^^^^^^^^^^^^^^^
OSError: libnvJitLink.so.13: cannot open shared object file: No such file or directory


Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!
transformers: 5.7.0 | huggingface_hub: 1.11.0 | torch: 2.11.0+cu130
Loaded env from /content/drive/MyDrive/MITS_secrets/.env


## Cell 3 — Decoding Configuration (зафиксирован)

Параметры применяются ОДИНАКОВО ко всем трём моделям (base, GSPO, KTO). Зафиксировано:

| Параметр | Значение | Обоснование |
|----------|----------|-------------|
| `num_predict` | 4096 | Покрывает 95-percentile thinking длин (GSPO учился с budget=2048; 4096 даёт запас на hard problems без overhead 8192). |
| `enable_thinking` | `True` | Матчит training distribution GSPO/KTO + native режим Qwen3.5-9B. False нивелировал бы RL-effect целиком — unfair. |
| `temperature` | 0.0 | Greedy для deterministic accuracy. Diversity sampling — Phase 1 (V-STaR). |
| `system_prompt` | `SYSTEM_PROMPT_CALC` | Apple-to-apple с предыдущими compare_base_vs_gspo отчётами. |

**Что это даёт для диплома**: section "Methodology — inference protocol" в одну таблицу. Альтернатива (запустить второй раз с `enable_thinking=False`) — рассматривается как _Appendix-grade ablation_, если останется compute после Phase 1-3.

In [2]:
# Cell 4: Decoding config (filled — single-protocol run)
# Same config applied to all three models (base, GSPO, KTO).

DECODING_CONFIG = {
    # 2048: GSPO trained budget=2048, covers 95-percentile thinking. 4096 был для
    # запаса на hard problems, но удваивает время генерации. 2048 = 2x speedup без
    # значимой потери (truncation risk <5%). Жертва ради скорости в lean-demo.
    'num_predict': 2048,
    # True: matches GSPO/KTO training distribution (chat_template_kwargs.enable_thinking
    # =True in grpo_qwen3_5_9b_(8).ipynb). Qwen3.5-9B base also natively supports <think>.
    # False would nullify the RL effect — unfair to GSPO/KTO. Keep one consistent mode.
    'enable_thinking': True,
    # 0.0: greedy decoding for accuracy eval. Diversity sampling is for V-STaR (Phase 1).
    'temperature': 0.0,
    # Same calc system prompt used in compare_base_vs_gspo_*.json — preserves apple-to-
    # apple with prior reports for the Cerebras-only path; only inference layer changes.
    'system_prompt': SYSTEM_PROMPT_CALC,
    'rationale': (
        'Single thinking-on protocol mirrors training distribution + production deployment. '
        '2048 budget covers GSPO training distribution. Greedy decoding for deterministic accuracy.'
    ),
}

assert all(v is not None for v in DECODING_CONFIG.values()), 'Fill in DECODING_CONFIG keys'
logger.info(f'Decoding config: {DECODING_CONFIG}')

In [3]:
# Cell 5: Load eval dataset — calc subset only (numeric / latex_boxed)
EVAL_PATH = PROJECT_ROOT / 'training/data/eval_dataset.jsonl'
all_problems = load_eval_dataset(str(EVAL_PATH))
calc_problems = [p for p in all_problems if p.get('answer_type') in ('numeric', 'latex_boxed')]
logger.info(f'Total: {len(all_problems)} | Calc subset: {len(calc_problems)}')
# Sanity
from collections import Counter
logger.info(f'By domain: {Counter(p["domain"] for p in calc_problems)}')
logger.info(f'By difficulty: {Counter(p["difficulty"] for p in calc_problems)}')

In [4]:
# Cell 5: Model loading via Unsloth (matches training stack from grpo_qwen3.5_9b.ipynb)
BASE_MODEL_ID = 'Qwen/Qwen3.5-9B'  # matches BASE_MODEL in training notebook
MAX_SEQ_LENGTH = 4096  # covers thinking budget 2048 + completion + headroom

ADAPTERS = {
    'base': None,
    'gspo': 'Siesher/mits-qwen3-9b-gspo',
    'kto':  'Siesher/mits-qwen3-9b-kto',
}


def load_model_with_adapter(adapter_id: str | None):
    """Load Qwen3.5-9B via Unsloth FastLanguageModel, optionally apply LoRA + merge.

    Why Unsloth: Qwen3.5-9B is image-text-to-text (multimodal) с Model Class
    AutoModelForImageTextToText. AutoModelForCausalLM не работает напрямую.
    FastLanguageModel.from_pretrained абстрагирует text-only branch loading,
    skipping vision tower (saves ~2GB VRAM + matches training stack exactly).

    Adapter merge_and_unload даёт plain HF model для inference path,
    FastLanguageModel.for_inference активирует Unsloth fast kernels.
    """
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=BASE_MODEL_ID,
        max_seq_length=MAX_SEQ_LENGTH,
        load_in_4bit=False,
        dtype=torch.bfloat16,
    )
    # Unsloth wraps tokenizer in some versions — unwrap if needed.
    if not hasattr(tokenizer, 'vocab_size') and hasattr(tokenizer, 'tokenizer'):
        tokenizer = tokenizer.tokenizer

    if adapter_id:
        model = PeftModel.from_pretrained(model, adapter_id)
        model = model.merge_and_unload()
        logger.info(f'Merged adapter {adapter_id}')

    FastLanguageModel.for_inference(model)
    return model, tokenizer


def generate_one(model, tokenizer, prompt: str) -> str:
    messages = [
        {'role': 'system', 'content': DECODING_CONFIG['system_prompt']},
        {'role': 'user', 'content': prompt},
    ]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True,
        enable_thinking=DECODING_CONFIG['enable_thinking'],
    )
    inputs = tokenizer(text, return_tensors='pt').to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=DECODING_CONFIG['num_predict'],
            do_sample=DECODING_CONFIG['temperature'] > 0,
            temperature=max(DECODING_CONFIG['temperature'], 1e-5),
            pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id,
        )
    completion = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    return completion


In [5]:
# Cell 7: Eval loop (Phase 0a — fast_mode=True, programmatic correctness only)
# Phase 0b async judge — отдельным скриптом потом, когда Cerebras quota свободна.
import re

def _normalize_for_compare(s) -> str:
    """Normalize answer string for numeric/symbolic comparison.

    Accepts str | int | float | None. eval_dataset.jsonl держит numeric
    ground_truth как int/float (JSON не coerce-ит в строки), поэтому
    str(s) делается до strip().
    """
    if s is None:
        return ''
    s = str(s).strip()
    if not s:
        return ''
    if s.startswith('$') and s.endswith('$'):
        s = s[1:-1].strip()
    s = re.sub(r'\\text\{[^}]*\}', '', s)
    s = re.sub(r'\\mathrm\{[^}]*\}', '', s)
    # Russian decimal comma → dot, strip whitespace incl. nbsp
    s = s.replace(' ', '').replace(' ', '').replace(',', '.')
    # Strip trailing punctuation
    s = s.rstrip('.;,')
    return s


def is_numeric_correct(extracted, ground_truth, tolerance: float = 0.02) -> bool | None:
    """Programmatic correctness check.

    - For numeric strings: relative tolerance 2% or absolute < 0.001 if truth ≈ 0.
    - For non-numeric: case-insensitive normalized string match.
    - Returns None if can't determine (empty extracted or both unparseable).
    """
    e_norm = _normalize_for_compare(extracted)
    g_norm = _normalize_for_compare(ground_truth)
    if not e_norm or not g_norm:
        return None
    try:
        ef, gf = float(e_norm), float(g_norm)
        if abs(gf) < 1e-9:
            return abs(ef) < 0.001
        return abs(ef - gf) / abs(gf) < tolerance
    except ValueError:
        return e_norm.lower() == g_norm.lower()


def eval_stage(stage_name: str, adapter_id: str | None, problems: List[Dict],
               fast_mode: bool = True) -> Dict[str, Any]:
    """Evaluate one stage.

    Args:
        fast_mode: Phase 0a (True) — programmatic correctness only, saves raw
                   completions для Phase 0b (async Cerebras judge).
                   Phase 0 original (False) — синхронный Cerebras judge per problem
                   (упирается в rate limits, не рекомендуется).
    """
    logger.info(f'=== Stage: {stage_name} ({adapter_id or "base"}) | fast_mode={fast_mode} ===')
    model, tokenizer = load_model_with_adapter(adapter_id)
    cerebras = None if fast_mode else CerebrasClient()
    completions = []
    t0 = time.time()
    for i, p in enumerate(problems):
        t_start = time.time()
        completion = generate_one(model, tokenizer, p['prompt'])
        t_gen = time.time() - t_start
        # Per-problem timing для первых 3 — диагностика fast-path vs fallback скорости.
        if i < 3:
            n_tokens = len(tokenizer.encode(completion))
            logger.info(f'  [{i+1}/{len(problems)}] gen={t_gen:.1f}s | tokens={n_tokens} | tok/s={n_tokens/max(t_gen,0.01):.1f}')
        extracted = extract_answer(completion)
        fmt = check_format_compliance(completion)
        visible = completion.split('</think>')[-1].strip() if '</think>' in completion else completion

        if fast_mode:
            correct = is_numeric_correct(extracted, p['ground_truth'])
            socratic, no_leak = None, None
        else:
            judge = evaluate_combined_quality(p['prompt'], visible, p['ground_truth'], cerebras)
            correct = judge.get('is_correct')
            socratic = judge.get('socratic_score')
            no_leak = judge.get('no_answer_leak')

        completions.append({
            'idx': i, 'domain': p['domain'], 'difficulty': p['difficulty'],
            'truth': p['ground_truth'], 'extracted': extracted,
            'correct': correct,
            'socratic_score': socratic,
            'no_answer_leak': no_leak,
            'completion_text': visible,  # saved для Phase 0b async judge
            **fmt,
        })

        log_every = 5 if fast_mode else 10
        if (i + 1) % log_every == 0:
            elapsed = time.time() - t0
            valid = [c for c in completions if c['correct'] is not None]
            acc = sum(1 for c in valid if c['correct']) / max(len(valid), 1)
            logger.info(f'  {i+1}/{len(problems)} | acc={acc:.3f} ({len(valid)} judged) | {elapsed/60:.1f} min')

    del model; torch.cuda.empty_cache()
    valid = [c for c in completions if c['correct'] is not None]
    accuracy = sum(1 for c in valid if c['correct']) / max(len(valid), 1)
    socratic_vals = [c['socratic_score'] for c in completions if c.get('socratic_score') is not None]
    leak_vals = [c['no_answer_leak'] for c in completions if c.get('no_answer_leak') is not None]
    return {
        'stage': stage_name, 'adapter': adapter_id, 'n': len(completions),
        'n_judged': len(valid),
        'accuracy': accuracy,
        'avg_socratic': (sum(socratic_vals) / len(socratic_vals)) if socratic_vals else None,
        'leak_rate': (sum(1 for v in leak_vals if v < 2) / len(leak_vals)) if leak_vals else None,
        'mode': 'fast_programmatic' if fast_mode else 'full_cerebras_judge',
        'completions': completions,
    }

In [6]:
# Cell 8: Run Phase 0a (fast_mode=True — programmatic correctness, no Cerebras).
# Phase 0b (async Cerebras judge over saved completions) — отдельный скрипт потом.
results = {}
for stage_name, adapter_id in ADAPTERS.items():
    results[stage_name] = eval_stage(stage_name, adapter_id, calc_problems, fast_mode=True)

report = {
    'protocol': 'honest_full_precision_phase0a',
    'mode': 'fast_programmatic',
    'note': (
        'Phase 0a: accuracy via programmatic numeric/string match (extract_answer vs '
        'ground_truth, 2% tolerance). socratic_score / leak_rate deferred to Phase 0b '
        '(async Cerebras judge over saved completions).'
    ),
    'timestamp': datetime.utcnow().isoformat(),
    'decoding_config': {k: v for k, v in DECODING_CONFIG.items() if k != 'system_prompt'},
    'system_prompt_hash': hash(DECODING_CONFIG['system_prompt']),
    'n_problems': len(calc_problems),
    'base': {k: v for k, v in results['base'].items() if k != 'completions'},
    'gspo': {k: v for k, v in results['gspo'].items() if k != 'completions'},
    'kto':  {k: v for k, v in results['kto'].items()  if k != 'completions'},
    'completions': {stage: r['completions'] for stage, r in results.items()},  # raw text сохранён
}

# Save to repo (so it can be committed) + Drive mirror (so it survives Colab teardown)
date_tag = datetime.utcnow().strftime('%Y%m%d')
out_repo  = PROJECT_ROOT / f'evaluation/reports/honest_full_precision_phase0a_{date_tag}.json'
out_drive = Path(f'/content/drive/MyDrive/MITS_secrets/honest_full_precision_phase0a_{date_tag}.json')
out_repo.parent.mkdir(parents=True, exist_ok=True)
out_repo.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding='utf-8')
if out_drive.parent.exists():
    out_drive.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding='utf-8')
    logger.info(f'Saved (Drive mirror): {out_drive}')
logger.info(f'Saved (repo): {out_repo}')

print('\n=== Phase 0a — Honest accuracy (full-precision bf16, identical decoding, programmatic correctness) ===')
for stage in ['base', 'gspo', 'kto']:
    r = results[stage]
    print(f"{stage:5s}  acc={r['accuracy']:.3f} ({r['n_judged']}/{r['n']} judged)")
print('\nNote: socratic_score / leak_rate — Phase 0b (run scripts/score_phase0b_async.py later).')

==((====))==  Unsloth 2026.5.1: Fast Qwen3_5 patching. Transformers: 5.7.0.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.494 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 8.0. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: QLoRA and full finetuning all not selected. Switching to 16bit LoRA.


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

[transformers] The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/760 [00:00<?, ?it/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/12.8M [00:00<?, ?B/s]

video_preprocessor_config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

KeyboardInterrupt: 

In [9]:
!pip show flash-linear-attention 2>&1 | head -5
try:
    import fla
    print(f'fla {fla.__version__} OK')
except Exception as e:
    print(f'fla import FAILED: {type(e).__name__}: {e}')

Name: flash-linear-attention
Version: 0.5.0
Summary: Fast linear attention models and layers
Home-page: https://github.com/fla-org/flash-linear-attention
Author: 
fla 0.5.0 OK


In [10]:
!nvcc --version 2>&1 | head -5
!which nvcc

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Fri_Feb_21_20:23:50_PST_2025
Cuda compilation tools, release 12.8, V12.8.93
Build cuda_12.8.r12.8/compiler.35583870_0
/usr/local/cuda/bin/nvcc


In [11]:
# 2. Try install causal-conv1d (pre-built wheel ИЛИ source build)
!pip install causal-conv1d --no-build-isolation 2>&1 | tail -20

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.7/180.7 kB 22.0 MB/s eta 0:00:00
  error: subprocess-exited-with-error
  
  × Building wheel for causal-conv1d (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel for causal-conv1d
Failed to build causal-conv1d
ERROR: ERROR: Failed to build installable wheels for some pyproject.toml based projects (causal-conv1d)


In [12]:
try:
    import causal_conv1d
    print(f'causal_conv1d {causal_conv1d.__version__} OK')
except Exception as e:
    print(f'causal_conv1d FAILED: {type(e).__name__}: {e}')

causal_conv1d FAILED: ModuleNotFoundError: No module named 'causal_conv1d'


In [14]:
!pip install -q "transformers>=5.0,<5.7"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 135.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
unsloth-zoo 2026.5.1 requires datasets!=4.0.*,!=4.1.0,<4.4.0,>=3.4.1, but you have datasets 4.8.5 which is incompatible.
unsloth-zoo 2026.5.1 requires torch<2.11.0,>=2.4.0, but you have torch 2.11.0 which is incompatible.
unsloth-zoo 2026.5.1 requires transformers!=4.52.0,!=4.52.1,!=4.52.2,!=4.52.3,!=4.53.0,!=4.54.0,!=4.55.0,!=4.55.1,!=4.57.4,!=4.57.5,!=5.0.0,!=5.1.0,<=5.5.0,>=4.51.3, but you have transformers 5.6.2 which is incompatible.
unsloth-zoo 2026.5.1 requires trl!=0.19.0,<=0.24.0,>=0.18.2, but you have trl 1.3.0 which is incompatible.
unsloth 2026.5.1 requires datasets!=4.0.*,!=4.1.0,<4.4.0,>=3.4.1, but you have datasets 4.8.5 which is incompatible.
unsloth 2026.5.1 requires torch<2.11.0,>=2.4.0, but you have torch 2.11.0 whic

In [1]:
import transformers
from transformers.models.auto.configuration_auto import CONFIG_MAPPING_NAMES
print(f"transformers: {transformers.__version__}")
print(f"qwen3_5 supported: {'qwen3_5' in CONFIG_MAPPING_NAMES}")

transformers: 5.6.2
qwen3_5 supported: True


In [2]:
!pip install -q "transformers==5.5.0"

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
unsloth-zoo 2026.5.1 requires datasets!=4.0.*,!=4.1.0,<4.4.0,>=3.4.1, but you have datasets 4.8.5 which is incompatible.
unsloth-zoo 2026.5.1 requires torch<2.11.0,>=2.4.0, but you have torch 2.11.0 which is incompatible.
unsloth-zoo 2026.5.1 requires trl!=0.19.0,<=0.24.0,>=0.18.2, but you have trl 1.3.0 which is incompatible.
unsloth 2026.5.1 requires datasets!=4.0.*,!=4.1.0,<4.4.0,>=3.4.1, but you have datasets 4.8.5 which is incompatible.
unsloth 2026.5.1 requires torch<2.11.0,>=2.4.0, but you have torch 2.11.0 which is incompatible.
unsloth 2026.5.1 requires trl!=0.19.0,<=0.24.0,>=0.18.2, but you have trl 1.3.0 which is incompatible.


In [1]:
import transformers
from transformers.models.auto.configuration_auto import CONFIG_MAPPING_NAMES
print(f"transformers: {transformers.__version__}")
print(f"qwen3_5 supported: {'qwen3_5' in CONFIG_MAPPING_NAMES}")

transformers: 5.5.0
qwen3_5 supported: True


In [2]:
!pip install -q --force-reinstall \
    "transformers==5.5.0" \
    "torch==2.10.0" "torchvision" "torchaudio" \
    "trl==0.24.0" "datasets==4.3.0" \
    --extra-index-url https://download.pytorch.org/whl/cu128

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.4/40.4 kB 4.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 7.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 MB 176.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 954.8/954.8 kB 245.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 247.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 706.8/706.8 MB 69.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 594.3/594.3 MB 74.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.1/193.1 MB 127.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.6/63.6 MB 180.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 267.5/267.5 MB 107.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 288.2/288.2 MB 109.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 287.2/287.2 MB 122.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━

In [1]:
!pip install -q causal-conv1d 2>&1 | tail -5

  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel for causal-conv1d
ERROR: ERROR: Failed to build installable wheels for some pyproject.toml based projects (causal-conv1d)


In [2]:
import subprocess
for pkg in ['torch', 'transformers', 'trl', 'datasets', 'causal_conv1d']:
    try:
        out = subprocess.check_output(['pip', 'show', pkg.replace('_', '-')], text=True)
        ver = [l for l in out.split('\n') if l.startswith('Version:')][0].split(': ')[1]
        print(f"{pkg}: {ver}")
    except Exception as e:
        print(f"{pkg}: NOT INSTALLED ({e})")

torch: 2.10.0+cu128
transformers: 5.5.0
trl: 0.24.0
datasets: 4.3.0
causal_conv1d: NOT INSTALLED (Command '['pip', 'show', 'causal-conv1d']' returned non-zero exit status 1.)


In [3]:
!pip uninstall -y cuda-toolkit cuda-bindings 2>&1 | tail -3

Found existing installation: cuda-bindings 12.9.4
Uninstalling cuda-bindings-12.9.4:
  Successfully uninstalled cuda-bindings-12.9.4


In [4]:
!pip install --no-build-isolation "causal-conv1d==1.4.0"

  Preparing metadata (setup.py) ... done
  Using cached ninja-1.13.0-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (5.1 kB)
Using cached ninja-1.13.0-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl (180 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 114.6 MB/s eta 0:00:00
  error: subprocess-exited-with-error
  
  × python setup.py bdist_wheel did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel for causal-conv1d
  Running setup.py clean for causal-conv1d
Failed to build causal-conv1d
ERROR: ERROR: Failed to build installable wheels for some pyproject.toml based projects (causal-conv1d)
